# DQA Anonymous Backbone MoE\n\nThis notebook runs the FedSTO-compatible DQA x MoE experiment with anonymous micro-experts inserted into the YOLOv5 backbone.\n\n- Dense warmup remains unchanged.\n- Phase 1 follows the FedSTO backbone update.\n- Phase 2 upcycles the dense checkpoint into backbone adapter MoE.\n- Experts are anonymous; DQA context conditions the router instead of assigning domain-specific expert names by hand.\n

In [ ]:
from pathlib import Path\n\nREPO = Path('/app/Object_Detection')\nRUNNER = REPO / 'dynamic_quality_aware_classwise_aggregation' / 'dqa_moe_trust_region' / 'scripts' / 'run_dqa_anonymous_backbone_moe.py'\nWORKSPACE = REPO / 'dynamic_quality_aware_classwise_aggregation' / 'dqa_moe_trust_region' / 'output' / '03_dqa_anonymous_backbone_moe'\nLOG = WORKSPACE / 'anonymous_backbone_moe.log'\nSUMMARY = WORKSPACE / 'anonymous_backbone_moe_round_summary.csv'\n\nWORKSPACE.mkdir(parents=True, exist_ok=True)\nRUNNER, WORKSPACE, LOG, SUMMARY\n

In [ ]:
cmd = [\n    'python', str(RUNNER),\n    '--workspace-root', str(WORKSPACE),\n    '--warmup-epochs', '50',\n    '--phase1-rounds', '20',\n    '--phase2-rounds', '20',\n    '--moe-start-phase', '2',\n    '--phase2-train-scope', 'all',\n    '--orthogonal-weight', '0.0001',\n    '--batch-size', '160',\n    '--phase2-batch-size', '64',\n    '--phase2-server-lr0', '0.001',\n    '--workers', '48',\n    '--gpus', '2',\n    '--master-port', '29547',\n    '--moe-num-experts', '4',\n    '--moe-top-k', '2',\n    '--moe-temperature', '1.0',\n    '--moe-scale', '0.25',\n    '--moe-shared-scale', '1.0',\n    '--moe-adapter-ratio', '0.125',\n    '--moe-levels', 'c3,c4,c5',\n    '--moe-kernels', '3,5,7',\n    '--moe-context-dim', '8',\n    '--moe-quality-dim', '4',\n    '--moe-router-noise-std', '0.01',\n    '--moe-balance-weight', '0.02',\n    '--moe-entropy-weight', '0.002',\n    '--moe-z-loss-weight', '0.0001',\n    '--moe-diversity-weight', '0.001',\n    '--moe-freeze-bn',\n    '--phase2-aggregate-lambda', '0.15',\n    '--phase2-max-relative-update', '0.01',\n    '--phase2-max-absolute-update', '0.0',\n    '--phase2-aggregate-scope', 'all',\n    '--router-diagnostic-split', 'cloudy',\n    '--router-diagnostic-images', '12',\n    '--run-final-eval',\n    '--final-eval-splits', 'cloudy,overcast,rainy,snowy,total',\n    '--val-batch-size', '32',\n    '--discord',\n]\n\nprint(' '.join(cmd))\n

In [ ]:
import subprocess\nfrom datetime import datetime\n\nwith LOG.open('a', encoding='utf-8') as f:\n    f.write(f'\\n\\n===== started {datetime.utcnow().isoformat()}Z =====\\n')\n    proc = subprocess.Popen(cmd, cwd=REPO, stdout=f, stderr=subprocess.STDOUT, text=True)\n\nprint(f'pid={proc.pid}')\nprint(f'log={LOG}')\n

In [ ]:
import pandas as pd\n\nif SUMMARY.exists():\n    display(pd.read_csv(SUMMARY).tail(20))\nelse:\n    print('summary is not created yet')\n\nrouter_csv = WORKSPACE / 'anonymous_backbone_moe_router_diagnostics.csv'\nif router_csv.exists():\n    display(pd.read_csv(router_csv).tail(20))\n\nif LOG.exists():\n    print(''.join(LOG.read_text(encoding='utf-8', errors='replace').splitlines(True)[-80:]))\n